In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [5]:
# Load the data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Check shape
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# First 5 rows
df.head()

Dataset shape: 7043 rows, 21 columns
Rows: 7,043
Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
# List all column names
print("Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

Column names:
 1. customerID
 2. gender
 3. SeniorCitizen
 4. Partner
 5. Dependents
 6. tenure
 7. PhoneService
 8. MultipleLines
 9. InternetService
10. OnlineSecurity
11. OnlineBackup
12. DeviceProtection
13. TechSupport
14. StreamingTV
15. StreamingMovies
16. Contract
17. PaperlessBilling
18. PaymentMethod
19. MonthlyCharges
20. TotalCharges
21. Churn


In [7]:
# Data types
print("\nData types:")
print(df.dtypes)


Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [8]:
# Statistical summary
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [9]:
# Check churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print("Churn distribution:")
for status, count in churn_counts.items():
    pct = churn_pct[status]
    print(f"  {status}: {count:,} customers ({pct:.1f}%)")

# Visualize
fig = px.bar(x=churn_counts.index, y=churn_counts.values,
             title='Churn Distribution',
             labels={'x': 'Churn Status', 'y': 'Number of Customers'},
             color=churn_counts.index)
fig.show()

Churn distribution:
  No: 5,174 customers (73.5%)
  Yes: 1,869 customers (26.5%)


In [10]:
# Export the churn distribution chart as HTML file
fig.write_html('churn_distribution.html')
print("Saved: churn_distribution.html")

Saved: churn_distribution.html


## Business Problem: Customer Churn Prediction

**What is churn?** Customers who cancel their service (Churn = "Yes").

**Business impact:** Acquiring new customers costs 5-10x more than retaining existing ones. Reducing churn by 5% can increase profits by 25-95%.

**Goal:** Predict which customers are likely to churn so the company can offer incentives before they leave.

**Success metric:** Achieve at least 75% recall on churned customers (identify 3 out of 4 who will leave).

**Dataset:** 7,043 customers, 21 features

**Target variable:** Churn (Yes = 1, No = 0)

**Class balance:** 26.5% churned, 73.5% did not churn (imbalanced)

---

## Interactive Charts

- [Churn Distribution](churn_distribution.html) - Click to view interactive bar chart

*Note: GitHub cannot preview large HTML files directly. Click the file, then click "Raw", then save and open locally.*

## Data Cleaning & Feature Engineering

This section handles real-world data issues:
- Fixing incorrect data types
- Handling missing values
- Creating new features
- Documenting all cleaning decisions

In [11]:
# Check data types
print("Current data types:")
print(df.dtypes)
print("\n" + "="*50)

# Specifically check TotalCharges
print(f"\nTotalCharges data type: {df['TotalCharges'].dtype}")
print(f"Sample values: {df['TotalCharges'].head().tolist()}")

Current data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


TotalCharges data type: str
Sample values: ['29.85', '1889.5', '108.15', '1840.75', '151.65']


In [12]:
# Convert TotalCharges from text to number
# First, check for empty strings or spaces
print("Before cleaning:")
print(f"  Unique values in TotalCharges: {df['TotalCharges'].nunique()}")
print(f"  Sample: {df['TotalCharges'].head(10).tolist()}")

# Convert to numeric (errors='coerce' turns invalid values into NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("\nAfter cleaning:")
print(f"  Data type: {df['TotalCharges'].dtype}")
print(f"  Sample: {df['TotalCharges'].head(10).tolist()}")

Before cleaning:
  Unique values in TotalCharges: 6531
  Sample: ['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95']

After cleaning:
  Data type: float64
  Sample: [29.85, 1889.5, 108.15, 1840.75, 151.65, 820.5, 1949.4, 301.9, 3046.05, 3487.95]


In [13]:
# Check for missing values in all columns
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_pct
})

# Show only columns with missing values
missing_df[missing_df['Missing Values'] > 0]

,Missing Values,Percentage (%)
TotalCharges,11,0.156183


In [14]:
# Check customers with missing TotalCharges
missing_customers = df[df['TotalCharges'].isnull()]
print(f"Customers with missing TotalCharges: {len(missing_customers)}")
print("\nTheir tenure values:")
print(missing_customers['tenure'].value_counts())

Customers with missing TotalCharges: 11

Their tenure values:
tenure
0    11
Name: count, dtype: int64


In [15]:
# Fill missing TotalCharges with 0 (customers with no tenure)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verify no more missing values
print(f"Missing values after fix: {df['TotalCharges'].isnull().sum()}")

Missing values after fix: 0


In [16]:
# Create average monthly charge (TotalCharges / tenure)
# For tenure = 0, set to MonthlyCharges
df['AvgMonthlyCharge'] = df.apply(
    lambda row: row['MonthlyCharges'] if row['tenure'] == 0 else row['TotalCharges'] / row['tenure'],
    axis=1
)

# Round to 2 decimal places
df['AvgMonthlyCharge'] = df['AvgMonthlyCharge'].round(2)

print("New feature created: AvgMonthlyCharge")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge']].head(10))

New feature created: AvgMonthlyCharge
   tenure  MonthlyCharges  TotalCharges  AvgMonthlyCharge
0       1           29.85         29.85             29.85
1      34           56.95       1889.50             55.57
2       2           53.85        108.15             54.08
3      45           42.30       1840.75             40.91
4       2           70.70        151.65             75.82
5       8           99.65        820.50            102.56
6      22           89.10       1949.40             88.61
7      10           29.75        301.90             30.19
8      28          104.80       3046.05            108.79
9      62           56.15       3487.95             56.26


In [17]:
# Convert Yes/No to 1/0
df['Churn_Num'] = (df['Churn'] == 'Yes').astype(int)

print("Churn conversion:")
print(df[['Churn', 'Churn_Num']].head(10))
print(f"\nChurn_Num distribution:")
print(df['Churn_Num'].value_counts())

Churn conversion:
  Churn  Churn_Num
0    No          0
1    No          0
2   Yes          1
3    No          0
4   Yes          1
5   Yes          1
6    No          0
7    No          0
8   Yes          1
9    No          0

Churn_Num distribution:
Churn_Num
0    5174
1    1869
Name: count, dtype: int64


In [18]:
# Final check
print("=== DATA CLEANING VERIFICATION ===\n")

print(f"1. TotalCharges data type: {df['TotalCharges'].dtype} (should be float64)")

print(f"\n2. Missing values:")
missing_final = df.isnull().sum().sum()
print(f"   Total missing values: {missing_final} (should be 0)")

print(f"\n3. New features created:")
new_features = ['AvgMonthlyCharge', 'Churn_Num']
for feat in new_features:
    print(f"   - {feat}: {df[feat].dtype}")

print(f"\n4. Dataset shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")

=== DATA CLEANING VERIFICATION ===

1. TotalCharges data type: float64 (should be float64)

2. Missing values:
   Total missing values: 0 (should be 0)

3. New features created:
   - AvgMonthlyCharge: float64
   - Churn_Num: int64

4. Dataset shape: (7043, 23)
   Rows: 7,043
   Columns: 23


In [19]:
# Save cleaned dataset for future use
df.to_csv('churn_data_cleaned.csv', index=False)
print("Saved: churn_data_cleaned.csv")

Saved: churn_data_cleaned.csv


## Data Cleaning Summary

### Issues Identified
1. **TotalCharges** was stored as text (object) instead of numbers
2. **11 missing values** in TotalCharges (customers with tenure = 0)

### Actions Taken
1. Converted TotalCharges to numeric using `pd.to_numeric()`
2. Filled missing TotalCharges with 0 (customers with no billing history)
3. Created new feature: **AvgMonthlyCharge** (TotalCharges / tenure)
4. Converted Churn (Yes/No) to numeric (1/0) for modeling

### Final Dataset
- **Shape:** 7,043 rows, 23 columns (added 2 new features)
- **Missing values:** 0
- **Data types:** All correct

### Next Steps
- Exploratory Data Analysis (EDA)
- Visualize churn patterns
- Build predictive model